In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

# -----------------------------
# 1. LOAD DATA
# -----------------------------
ratings = pd.read_csv("Dataset/rating.csv")  # userId, movieId, rating
ratings = ratings.iloc[:100000]

# -----------------------------
# 2. CREATE USER-MOVIE MATRIX
# -----------------------------
user_item = ratings.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
).fillna(0)

user_ids = user_item.index.tolist()
movie_ids = user_item.columns.tolist()

matrix = user_item.values
matrix = torch.tensor(matrix, dtype=torch.float32)

num_users, num_movies = matrix.shape

# -----------------------------
# 3. AUTOENCODER MODEL
# -----------------------------
class AutoEncoder(nn.Module):
    def __init__(self, num_movies):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(num_movies, 256),
            nn.ReLU(),
            nn.Linear(256, 64),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.Linear(64, 256),
            nn.ReLU(),
            nn.Linear(256, num_movies)
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

model = AutoEncoder(num_movies)

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# -----------------------------
# 4. TRAINING
# -----------------------------
epochs = 10

for epoch in range(epochs):
    model.train()

    output = model(matrix)

    loss = loss_fn(output, matrix)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# -----------------------------
# 5. RECOMMENDATION FUNCTION
# -----------------------------
def recommend_autoencoder(user_id, top_k=10):
    model.eval()

    user_idx = user_ids.index(user_id)

    user_vector = matrix[user_idx].unsqueeze(0)

    with torch.no_grad():
        preds = model(user_vector).numpy().flatten()

    # remove already rated movies
    seen = matrix[user_idx].numpy()
    preds[seen > 0] = -np.inf

    top_indices = np.argsort(preds)[-top_k:][::-1]

    return [movie_ids[i] for i in top_indices]

# -----------------------------
# 6. HIT RATE@10 EVALUATION
# -----------------------------
K = 10
hits = []

for user_idx in range(num_users):

    user_vector = matrix[user_idx].numpy()
    liked_indices = np.where(user_vector >= 4)[0]

    if len(liked_indices) < 2:
        continue

    train_idx = liked_indices[:-1]
    test_idx = liked_indices[-1]

    input_vec = user_vector.copy()
    input_vec[test_idx] = 0  # hide test movie

    input_tensor = torch.tensor(input_vec).unsqueeze(0)

    with torch.no_grad():
        preds = model(input_tensor).numpy().flatten()

    preds[train_idx] = -np.inf

    top_k = np.argsort(preds)[-K:][::-1]

    hits.append(1 if test_idx in top_k else 0)

print("Hit Rate@10:", np.mean(hits))

Epoch 1, Loss: 0.2351
Epoch 2, Loss: 0.2331
Epoch 3, Loss: 0.2308
Epoch 4, Loss: 0.2274
Epoch 5, Loss: 0.2233
Epoch 6, Loss: 0.2189
Epoch 7, Loss: 0.2142
Epoch 8, Loss: 0.2097
Epoch 9, Loss: 0.2057
Epoch 10, Loss: 0.2023
Hit Rate@10: 0.04154727793696275
